# 网络

## 简介

本教程概述了 **skrf** 的微波网络分析功能。在本教程以及 scikit-rf 文档的其余部分中，假设 **skrf** 已作为 `rf` 导入。你是否在自己的代码中遵循此约定取决于你自己。

In [ ]:
import numpy as np

import skrf as rf

如果这产生了导入错误，请参阅[安装](Installation.ipynb)。

## 创建网络

**skrf** 提供了一个 N 端口微波 [Network](../api/network.rst) 对象。可以通过多种方式创建 [Network](../api/network.rst)：
 - 从 Touchstone 文件
 - 从 S 参数
 - 从 Z 参数
 - 从其他射频参数（Y、ABCD、T 等）
 
下面给出了每种情况的一些示例。

### 从 Touchstone 文件创建网络
[Touchstone 文件](https://en.wikipedia.org/wiki/Touchstone_file)（`.sNp` 文件，`N` 为端口数）是导出 N 端口网络参数数据和线性有源器件、无源滤波器、无源器件或互连网络的噪声数据的_事实_标准。从 Touchstone 文件创建网络很简单：

In [ ]:
from skrf import Frequency, Network

ring_slot = Network('data/ring slot.s2p')

请注意，某些软件（如 ANSYS HFSS）会向 Touchstone 标准添加额外信息，例如注释、仿真参数、端口阻抗或 Gamma（波数）。如果检测到这些数据，它们也会被导入。

如果在命令行中输入网络对象，会打印出简短的描述

In [ ]:
ring_slot

### 从 s 参数创建网络
网络也可以通过直接传递 `frequency`、`s` 参数（以及可选的端口阻抗 `z0`）的值来创建。

N 端口网络的散射矩阵预期是一个形状为 `(nb_f, N, N)` 的 Numpy 数组，其中 `nb_f` 是频率点的数量，`N` 是网络的端口数。

<img src="figures/arrays_s_vs_f.svg" width="300">

In [ ]:
# dummy 2-port network from Frequency and s-parameters
freq = Frequency(1, 10, 101, 'ghz')
rng = np.random.default_rng()
s = rng.uniform(size=(101, 2, 2)) + 1j*rng.uniform(size=(101, 2, 2))  # random complex numbers
# if not passed, will assume z0=50. name is optional but it's a good practice.
ntwk = Network(frequency=freq, s=s, name='random values 2-port')
ntwk

In [ ]:
ntwk.plot_s_db()

通常，s 参数存储在单独的数组中。在这种情况下，需要构造 s 矩阵：

In [ ]:
# let's assume we have separate arrays for the frequency and s-parameters
f = np.array([1, 2, 3, 4]) # in GHz
S11 = rng.uniform(size=4)
S12 = rng.uniform(size=4)
S21 = rng.uniform(size=4)
S22 = rng.uniform(size=4)

# Before creating the scikit-rf Network object, one must forge the Frequency and S-matrix:
freq2 = rf.Frequency.from_f(f, unit='GHz')

# forging S-matrix as shape (nb_f, 2, 2)
# there is probably smarter way, but less explicit for the purpose of this example:
s = np.zeros((len(f), 2, 2), dtype=complex)
s[:,0,0] = S11
s[:,0,1] = S12
s[:,1,0] = S21
s[:,1,1] = S22

# constructing Network object
ntw = rf.Network(frequency=freq2, s=s)

print(ntw)

如有必要，特性阻抗可以作为标量（对所有频率相同）、列表或数组传递：

In [ ]:
ntw2 = rf.Network(frequency=freq, s=s, z0=25, name='same z0 for all ports')
print(ntw2)
ntw3 = rf.Network(frequency=freq, s=s, z0=[20, 30], name='different z0 for each port')
print(ntw3)
ntw4 = rf.Network(frequency=freq, s=s, z0=rng.uniform(size=(4,2)), name='different z0 for each frequencies and ports')
print(ntw4)

### 从 z 参数

由于网络也可以从它们的 Z 参数定义，Network 有 `from_z()` 方法：

In [ ]:
# 1-port network example
z = np.full((len(freq), 1, 1), 10j)  # replicate z=10j for all frequencies

ntw = rf.Network(frequency=freq, z=z)
print(ntw)

### 从其他网络参数：Y、ABCD、H、T
也可以从其他类型的射频参数生成网络：[Y](https://en.wikipedia.org/wiki/Two-port_network#Admittance_parameters_(y-parameters))、[ABCD](https://en.wikipedia.org/wiki/Two-port_network#ABCD-parameters)、[H](https://en.wikipedia.org/wiki/Two-port_network#Hybrid_parameters_(h-parameters)) 或 [T](https://en.wikipedia.org/wiki/Two-port_network#Scattering_transfer_parameters_(T-parameters))，分别在创建 `Network` 时使用 `y=`、`a=`、`h=` 或 `t=` 参数。

例如，串联阻抗的 [ABCD 参数](https://en.wikipedia.org/wiki/Two-port_network#ABCD-parameters) 为：
$$
\left[
\begin{array}{cc}
1 & Z \\
0 & 1
\end{array}
\right]
$$
因此可以像这样创建相关的网络：

In [ ]:
z = 20
abcd = np.array([[1, z],
              [0, 1]])

a = np.tile(abcd, (len(freq),1,1))
ntw = Network(frequency=freq, a=a)
print(ntw)

请注意，如果需要，还可以使用便利函数将某些参数转换为另一种参数：

| from\to |   S   |   Z   |   Y   |  ABCD |   T   |   H   |
|:-------:|:-----:|:-----:|:-----:|:-----:|:-----:|:-----:|
|    S    |       | `s2z` | `s2y` | `s2a` | `s2t` | `s2h` |
|    Z    | `z2s` |       | `z2y` | `z2a` | `z2t` | `z2h` |
|    Y    | `y2s` | `y2z` |       |       | `y2t` |       |
|   ABCD  | `a2s` | `a2z` |       |       |       |       |
|    T    | `t2s` | `t2z` | `t2y` |       |       |       |
|    H    | `h2s` | `h2z` |       |       |       |       |

In [ ]:
# example: converting a -> s
s = rf.network.a2s(a)
# checking that these S-params are the same
np.all(ntw.s == s)

## 基本属性
	
微波 [Network](../api/network.rst) 的基本属性由以下属性提供：

* `Network.s` : 散射参数矩阵。
* `Network.z0`  : 端口特性阻抗矩阵。
* `Network.frequency`  : 频率对象。

[Network](../api/network.rst) 对象还有许多其他属性和方法。如果你使用 IPython，那么这些属性和方法可以在命令行上通过"Tab"键补全。


	In [1]: ring_slot.s<TAB>
	ring_slot.line.s              ring_slot.s_arcl         ring_slot.s_im
	ring_slot.line.s11            ring_slot.s_arcl_unwrap  ring_slot.s_mag
	...

所有网络参数在内部都表示为复数 `numpy.ndarray`。s 参数的形状为 `(nfreq, nport, nport)`：

In [ ]:
np.shape(ring_slot.s)

## 切片

你可以以任何方式对 `Network.s` 属性进行切片。

In [ ]:
ring_slot.s[:11,1,0]  # get first 10 values of S21

也可以直接在 Network 对象上按频率进行切片

In [ ]:
ring_slot[0:10] #  Network for the first 10 frequency points

或者使用人类友好的字符串，

In [ ]:
ring_slot['80-90ghz']

请注意，直接在 Network 上切片**返回一个 Network**。因此，在两个维度上表达切片的一个好方法是

In [ ]:
ring_slot.s11['80-90ghz']

## 绘图

除其他外，[Network](../api/network.rst) 类的方法提供了绘制网络参数各分量的便捷方式：

* `Network.plot_s_db` : 以对数刻度绘制 s 参数的幅值
* `Network.plot_s_deg` : 以度为单位绘制 s 参数的相位
* `Network.plot_s_smith` : 在史密斯圆图上绘制复数 s 参数
* ...

如果你想使用 skrf 的绘图样式，

In [ ]:
%matplotlib inline
rf.stylely()

	
要在史密斯圆图上绘制 `ring_slot` 的所有四个 s 参数。

In [ ]:
ring_slot.plot_s_smith()

结合切片功能，

In [ ]:
from matplotlib import pyplot as plt

plt.title('Ring Slot $S_{21}$')

ring_slot.s11.plot_s_db(label='Full Band Response')
ring_slot.s11['82-90ghz'].plot_s_db(lw=3,label='Band of Interest')

有关绘图的详细信息，请参阅 [Plotting](Plotting.ipynb)。

## 算术运算
	
通过重载运算符可以访问对散射参数矩阵的逐元素数学运算。为了说明它们的用法，从 `data` 模块加载几个存储的网络。

In [ ]:
from skrf.data import wr2p2_delayshort as delayshort
from skrf.data import wr2p2_short as short

short - delayshort
short + delayshort
short * delayshort
short / delayshort


所有这些操作都返回 [Network](../api/network.rst) 类型。例如，要绘制 `short` 和 `delay_short` 之间的复数差异，

In [ ]:
difference = (short - delayshort)
difference.plot_s_mag(label='Mag of difference')

另一个常见应用是使用除法运算符计算相位差，

In [ ]:
(delayshort/short).plot_s_deg(label='Detrended Phase')

线性运算符也可以与标量或与 [Network](../api/network.rst) 长度相同的 `numpy.ndarray` 一起使用。

In [ ]:
hopen = (short*-1)
hopen.s[:3,...]

In [ ]:
rando =  hopen *rng.uniform(size=len(hopen))
rando.s[:3,...]

## 网络比较
比较运算符也适用于网络：

In [ ]:
short == delayshort

In [ ]:
short != delayshort

## 级联和去嵌入

2 端口网络的级联和去嵌入也可以通过运算符完成。`cascade` 函数可以通过幂运算符 `**` 调用。要计算两个单独网络 `line` 和 `short` 级联连接后的新网络，

In [ ]:
short = rf.data.wr2p2_short
line = rf.data.wr2p2_line
delayshort = line ** short

去嵌入可以通过级联网络的*逆*来完成。网络的逆通过属性 `Network.inv` 访问。要从 `delay_short` 中去嵌入 `short`，

In [ ]:
short_2 = line.inv ** delayshort

short_2 == short

级联运算符也适用于 2N 端口网络。这在[关于平衡网络的示例](../examples/networktheory/Balanced%20Network%20De-embedding.ipynb)中有所说明。例如，假设你要级联三个 4 端口网络 `ntw1`、`ntw2` 和 `ntw3`，你可以使用：
`resulting_ntw = ntw1 ** ntw2 ** ntw3`。请注意，`**` 级联运算符对 4 端口网络假设的端口方案是：

```
   ntw1    ntw2    ntw3
  +----+  +----+  +----+
0-|0  2|--|0  2|--|0  2|-2
1-|1  3|--|1  3|--|1  3|-3
  +----+  +----+  +----+
```

有关连接网络的更多文档可在此处获取：[连接网络](./Connecting_Networks.ipynb)。

## 连接多端口

**skrf** 支持连接 N 端口网络的任意端口。它使用一种称为子网络增长的算法[[1]](#References)，通过函数 `connect()` 实现。

作为一个示例，终止理想 3 路分配器的一个端口可以这样做，

In [ ]:
tee = rf.data.tee
tee

要将 tee 的端口 `1` 连接到延迟短路的端口 `0`，

In [ ]:
terminated_tee = rf.network.connect(tee, 1, delayshort, 0)
terminated_tee

请注意，此函数会考虑端口阻抗。如果两个连接的端口具有不同的端口阻抗，则会插入适当的阻抗不匹配。

有关连接网络的更多信息可在此处获取：[连接网络](./Connecting_Networks.ipynb)。

对于许多任意 N 端口网络之间的高级连接，`Circuit` 对象更相关，因为它允许明确端口和网络之间的连接。有关更多信息，请参阅 [Circuit 文档](Circuit.ipynb)。

## 插值和连接

改变 [Network](../api/network.rst) 频率点数是一个常见的需求。要使用运算符和级联函数，所涉及的网络必须具有匹配的频率，例如。如果两个网络具有不同的频率信息，则会引发错误，

In [ ]:
from skrf.data import wr2p2_line1 as line1

line1

    line1+line
    
    ---------------------------------------------------------------------------
    IndexError                                Traceback (most recent call last)
    <ipython-input-49-82040f7eab08> in <module>()
    ----> 1 line1+line

    /home/alex/code/scikit-rf/skrf/network.py in __add__(self, other)
        500 
        501         if isinstance(other, Network):
    --> 502             self.__compatible_for_scalar_operation_test(other)
        503             result.s = self.s + other.s
        504         else:

    /home/alex/code/scikit-rf/skrf/network.py in __compatible_for_scalar_operation_test(self, other)
        701         '''
        702         if other.frequency  != self.frequency:
    --> 703             raise IndexError('Networks must have same frequency. See `Network.interpolate`')
        704 
        705         if other.s.shape != self.s.shape:

    IndexError: Networks must have same frequency. See `Network.interpolate`


	
使用 `Network.resample` 沿频率轴对其中一个网络进行插值可以解决这个问题。

In [ ]:
line1.resample(201)
line1

现在我们可以做事情了

In [ ]:
line1 + line

你也可以从 `Frequency` 对象进行插值。例如，

In [ ]:
line.interpolate(line1.frequency)

相关应用是需要组合覆盖不同频率范围的网络。两个网络可以使用 `stitch` 连接（又名拼接）在一起，它沿频率轴连接网络。要组合 WR-2.2 网络和 WR-1.5 网络，

In [ ]:
from skrf.data import wr1p5_line, wr2p2_line

big_line = rf.network.stitch(wr2p2_line, wr1p5_line)
big_line

## 读写


对于长期数据存储，**skrf** 支持读取和部分支持写入 [touchstone 文件格式](http://en.wikipedia.org/wiki/Touchstone_file)。读取通过上面显示的 Network 初始化程序完成，写入则使用 `Network.write_touchstone()` 方法。

对于**临时**数据存储，**skrf** 对象可以使用函数 `skrf.io.general.read` 和 `skrf.io.general.write` 进行 [pickle](http://docs.python.org/2/library/pickle.html)。使用临时 pickle 而不是 touchstone 的原因是它们存储网络的所有属性，而 touchstone 文件仅存储部分信息。

In [ ]:
rf.io.write('data/myline.ntwk',line) # write out Network using pickle

In [ ]:
ntwk = Network('data/myline.ntwk') # read Network using pickle

通常有一整个目录的文件需要分析。`rf.read_all` 可以快速从目录中的所有文件创建网络。要加载 `data/` 目录中包含字符串 `'wr2p2'` 的所有 **skrf** 文件。

In [ ]:
dict_o_ntwks = rf.io.read_all(rf.data.pwd, contains = 'wr2p2')
dict_o_ntwks

其他时候你知道需要分析的文件列表。`rf.read_all` 也接受 files 参数。此示例文件列表仅包含同一目录中的文件，但你可以根据应用程序的需要存储文件。

In [ ]:
import os

dict_o_ntwks_files = rf.io.read_all(
    files=[os.path.join(rf.data.pwd, test_file) for test_file in ['ntwk1.s2p', 'ntwk2.s2p']]
)
dict_o_ntwks_files

## 其他参数

本教程侧重于 s 参数，但其他网络表示也可用。阻抗和导纳参数分别可以通过参数 `Network.z` 和 `Network.y` 访问。复数参数的标量分量，如 `Network.z_re`、`Network.z_im` 和绘图方法也可用。

其他参数仅适用于 2 端口网络，例如波级联参数（`Network.t`）、ABCD 参数（`Network.a`）或混合参数（`Network.h`）。

In [ ]:
ring_slot.z[:3,...]

In [ ]:
ring_slot.plot_z_im(m=1,n=0)

## 参考文献


[1] Compton, R.C.; , "Perspectives in microwave circuit analysis," Circuits and Systems, 1989., Proceedings of the 32nd Midwest Symposium on , vol., no., pp.716-718 vol.2, 14-16 Aug 1989. URL: http://ieeexplore.ieee.org/stamp/stamp.jsp?tp=&arnumber=101955&isnumber=3167